# AI Tic-Tac-Toe Demo (CPTS 440)

**Project:** Configurable n×n Tic-Tac-Toe AI using Minimax, Alpha-Beta Pruning, Q-Learning, and Neural Network Heuristics  
**Authors:** Aman Verma, Nicholas Vendeland, Jason  
**Course:** CPTS 440, Washington State University  
**GitHub:** [https://github.com/amanverma-wsu/440-Project](https://github.com/amanverma-wsu/440-Project)

---

This guided Colab notebook demonstrates how to download, install, run, and evaluate the AI Tic-Tac-Toe project.

## Demo Goals
1. Show how to clone and install the project
2. Run the benchmark comparison
3. Reproduce the main experiment outputs
4. Display generated result plots
5. Run direct tests for each AI agent (Alpha-Beta, Q-Learning, Neural Net)
6. Summarize the main empirical findings

---
## Step 1: Clone the GitHub Repository

This downloads the project source code into the Colab environment.

In [ ]:
import os

REPO_URL = "https://github.com/amanverma-wsu/440-Project.git"
REPO_DIR = "440-Project"

if os.path.isdir(REPO_DIR):
    print(f"'{REPO_DIR}' already exists — skipping clone.")
else:
    !git clone {REPO_URL}

%cd {REPO_DIR}
print(f"\n✅ Working directory: {os.getcwd()}")

## Step 2: Install Dependencies

We install only the packages needed for the Colab demo (NumPy, PyTorch, Matplotlib).  
Flask and Pygame are only required for the local web/GUI interfaces and are **not** installed here to avoid unnecessary errors in a headless Colab environment.

In [ ]:
!pip install -q numpy torch matplotlib

# Verify installations
import sys, numpy, torch, matplotlib
print(f"Python:     {sys.version.split()[0]}")
print(f"NumPy:      {numpy.__version__}")
print(f"PyTorch:    {torch.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print("\n✅ All Colab dependencies installed successfully.")

## Step 3: Verify Project Files

Before running the code, we confirm that all important project files are present.

| File | Purpose |
|---|---|
| `board.py` | Board representation and win detection |
| `ai.py` | Minimax, Alpha-Beta, and Random agents |
| `heuristic.py` | Heuristic evaluation logic |
| `qlearning.py` | Q-Learning agent |
| `nn_heuristic.py` | Neural network heuristic |
| `benchmark.py` | Benchmark comparison script |
| `experiments.py` | Full experiment suite |

In [ ]:
required_files = [
    "board.py", "ai.py", "heuristic.py",
    "qlearning.py", "nn_heuristic.py",
    "benchmark.py", "experiments.py"
]

missing = [f for f in required_files if not os.path.isfile(f)]

if missing:
    print("❌ Missing files:", missing)
    print("Please check the repository or re-run Step 1.")
else:
    print("✅ All required files are present.")
    !ls -la *.py

---
## Step 4: Run the Benchmark Comparison

This is the **core demo**. The benchmark compares Minimax, Alpha-Beta Pruning, and Q-Learning on a 3×3 board.

### What to observe
- Win / draw / loss results
- Average nodes explored per move
- Average time per move
- Alpha-Beta node reduction and speedup

**Expected result:** Alpha-Beta should explore ~96% fewer nodes than Minimax while preserving identical decision quality.

In [ ]:
!python benchmark.py

### Benchmark Explanation

The benchmark demonstrates the main project insight:

- **Minimax** searches the game tree exhaustively — correct but slow (~179,823 nodes/move).
- **Alpha-Beta pruning** makes the same decision but skips irrelevant branches (~6,874 nodes/move).
- **Q-Learning** is extremely fast at inference (table lookup, ~1 node) but may not be as reliable as full adversarial search.

The most important result is that **Alpha-Beta reduces node exploration by ~96%** compared to plain Minimax.

---
## Step 5: Run the Full Experiment Suite

This runs the larger set of experiments used in the report and slides.

Experiments include:
1. Correctness verification on 3×3
2. Minimax vs Alpha-Beta node comparison
3. Depth limit impact on 4×4 boards
4. Board-size scaling analysis
5. Q-Learning training curve
6. Q-Learning vs Alpha-Beta comparison
7. Neural-network heuristic training and evaluation

**⏱ Note:** This cell may take **several minutes** because it trains models and generates plots.

In [ ]:
!python experiments.py

## Step 6: List Generated Result Files

The experiment script saves graphs and model outputs inside the `results/` folder.

In [ ]:
if os.path.isdir("results"):
    files = os.listdir("results")
    print(f"✅ Found {len(files)} files in results/:")
    for f in sorted(files):
        size = os.path.getsize(os.path.join("results", f))
        print(f"  {f:40s}  ({size:,} bytes)")
else:
    print("❌ results/ folder not found. Run Step 5 (experiments.py) first.")

## Step 7: Display Generated Plots

This cell displays all `.png` images generated by the experiments.  
These plots visually support the report results, including node reduction, scaling, Q-Learning training, and neural-network comparisons.

In [ ]:
from IPython.display import Image, display

results_dir = "results"

if os.path.isdir(results_dir):
    image_files = sorted([
        f for f in os.listdir(results_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])
    if image_files:
        for img in image_files:
            print(f"\n--- {img} ---")
            display(Image(filename=os.path.join(results_dir, img), width=700))
    else:
        print("No plot images found in results/.")
else:
    print("results/ folder does not exist yet. Run experiments.py first.")

---
## Step 8: Direct Alpha-Beta Agent Test

This creates a 3×3 board and asks the Alpha-Beta agent to select its first move.  
This shows the AI working directly without running the full interactive game.

**Output:** Initial board state, selected move, nodes explored, and time taken.

In [ ]:
from board import Board
from ai import AlphaBetaAgent

board = Board(3)
agent = AlphaBetaAgent(Board.X)

move = agent.get_move(board)

print("Initial Board:")
print(board)
print(f"\nAlpha-Beta Selected Move: {move}")
print(f"Nodes Explored:           {agent.stats.nodes_explored}")
print(f"Time Taken:               {agent.stats.elapsed_time:.4f}s")

## Step 9: Direct Q-Learning Agent Demo

This cell trains a Q-Learning agent from scratch using the built-in `train()` method over 5,000 episodes, then evaluates it against a random opponent.  
This demonstrates the reinforcement learning pipeline interactively.

**Output:** Training summary, sample move, and win rate vs Random.

In [ ]:
from board import Board
from qlearning import QLearningAgent
from ai import RandomAgent

# Create and train a Q-Learning agent for 5,000 episodes (quick demo)
q_agent = QLearningAgent(Board.X, epsilon=0.3)

print("Training Q-Learning agent for 5,000 episodes...")
rewards = q_agent.train(num_episodes=5000, board_size=3, decay_epsilon=True)

print(f"\n✅ Training complete!")
print(f"   Q-table entries:         {len(q_agent.q_table):,}")
print(f"   Final cumulative reward: {rewards[-1]:.1f}")

# Test: trained agent picks a move on an empty board
test_board = Board(3)
q_move = q_agent.get_move(test_board)
print(f"\n   Sample move on empty 3×3 board: {q_move}")
print(f"   Inference time: {q_agent.stats.elapsed_time:.6f}s (table lookup)")

# Evaluation: play 200 games vs Random
wins, draws, losses = 0, 0, 0
for _ in range(200):
    b = Board(3)
    opp = RandomAgent(Board.O)
    while not b.is_terminal():
        if b.current_player() == Board.X:
            mv = q_agent.get_move(b)
        else:
            mv = opp.get_move(b)
        if mv:
            b.make_move(mv[0], mv[1], b.current_player())
    winner = b.check_winner()
    if winner == Board.X:
        wins += 1
    elif winner == Board.O:
        losses += 1
    else:
        draws += 1

print(f"\n   Evaluation vs Random (200 games): {wins}W / {draws}D / {losses}L  ({wins/2:.1f}% win rate)")

## Step 10: Direct Neural Network Heuristic Demo

This cell trains a small neural network to approximate minimax board evaluations using `train_nn()`, then uses the learned heuristic via `make_nn_heuristic()` to evaluate board positions.  
This demonstrates the learned evaluation pipeline interactively.

**Output:** Training loss summary and a comparison of NN-heuristic vs handcrafted-heuristic board scores.

In [ ]:
import numpy as np
import torch
from board import Board
from nn_heuristic import train_nn, make_nn_heuristic
from heuristic import evaluate_board

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Train the neural network on 3×3 minimax-labeled positions
print("=" * 50)
print("Training Neural Network Heuristic (3×3)")
print("=" * 50)
model, train_losses, val_losses = train_nn(board_size=3, epochs=300, verbose=True)

print(f"\n  Final train loss: {train_losses[-1]:.6f}")
print(f"  Final val loss:   {val_losses[-1]:.6f}")

# Compare NN heuristic vs handcrafted heuristic on a sample board
nn_heuristic_fn = make_nn_heuristic(board_size=3)

board = Board(3)
board.make_move(1, 1, Board.X)  # X takes center
board.make_move(0, 0, Board.O)  # O takes corner

print(f"\nTest board (X at center, O at corner):")
print(board)

nn_score = nn_heuristic_fn(board, Board.X, Board.O)
hc_score = evaluate_board(board, Board.X, Board.O)

print(f"\n  Neural Net heuristic score:    {nn_score:+.3f}")
print(f"  Handcrafted heuristic score:   {hc_score:+.3f}")
print(f"  Both agree X has advantage:    {'Yes ✅' if nn_score > 0 and hc_score > 0 else 'No'}")

---
## Step 11: Optional — Local Interfaces

The following interfaces require a local machine (not Colab) because they need a display or a local server.

### Command-Line Game
```bash
pip install numpy torch matplotlib flask pygame
python game.py
```
The CLI lets the user choose board size, AI algorithm, and player order.

### Flask Web App
```bash
python web_app.py
# Then open http://127.0.0.1:5000/
```
The web app provides a browser-based interface for playing the game with options for CPU mode and 2-player mode.

### Pygame GUI
```bash
python gui.py
```
The graphical interface allows visual gameplay with menu selection.

---
## Final Results Summary

| Algorithm | Avg Nodes/Move | Avg Time/Move | Main Strength | Main Weakness |
|---|---|---|---|---|
| Minimax | 179,823 | ~1.81s | Optimal on small boards | Exhaustive — very slow |
| Alpha-Beta | 6,874 | ~0.08s | Same quality, 96% fewer nodes | Still expensive on larger boards |
| Q-Learning | 1 (lookup) | ~0.000005s | Instant inference | Reliability depends on training |
| NN Heuristic | Varies | ~0.068s | Learns from data, adaptable | Depends on training labels |

## Key Takeaways

1. **Minimax** is a strong baseline because it searches the full game tree.
2. **Alpha-Beta pruning** is the most important optimization — it reduces unnecessary search by ~96%.
3. **Larger boards** require depth limits and heuristic evaluation; full-depth search is infeasible.
4. **Q-Learning** shifts computation from inference to training time, enabling instant moves.
5. **Neural networks** can learn evaluation functions from data, replacing hand-coded rules.
6. The project demonstrates the **trade-off between optimality, speed, and scalability**.

## Demo Completion Checklist

| Requirement | Status |
|---|---|
| How to download/install the code | ✅ Steps 1–3 |
| Step-by-step instructions for running | ✅ Steps 4–10 |
| Producing and displaying results | ✅ Steps 5–7 |
| Direct agent demos (Alpha-Beta, Q-Learning, NN) | ✅ Steps 8–10 |
| Local interface instructions (CLI, Web, GUI) | ✅ Step 11 |